# 01 — Pilot 2: YouTube comment collection

**FYP: Vietnamese sarcasm/irony detection**  
Purpose: collect **public top-level YouTube comments**, clean/deduplicate the text, then produce **two independent annotation sheets** for Pilot 2.

**Before collecting:** follow your university ethics approval and YouTube API Terms. Do not publish raw comment dumps, user identifiers, or your API key. This is **pilot data**, not a labeled training dataset. Online news comments can be added via a separately approved workflow.

**Recommended Pilot 2:** select 3–5 public Vietnamese-language videos and aim for about 150–200 raw comments before filtering; sample 30–50 for double annotation. Do not artificially balance labels before independent annotation.

## 1. Imports and paths

Run this with the `fyp-sarcasm` kernel. Install missing packages from your `requirements.txt` in your terminal if needed: `python -m pip install -r requirements.txt`.

In [ ]:
import os
import re
from pathlib import Path
from urllib.parse import urlparse, parse_qs
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Compatible with VS Code using either the repository root or notebooks/ as cwd.
cwd = Path.cwd().resolve()
root_candidates = [cwd, *list(cwd.parents)]
PROJECT_ROOT = next((p for p in root_candidates if (p / "requirements.txt").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find repository root containing requirements.txt")

ENV_FILE = PROJECT_ROOT / ".env"
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"Create a private .env file at {ENV_FILE}")
load_dotenv(ENV_FILE, override=True)
API_KEY = os.getenv("YOUTUBE_API_KEY", "").strip()
if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY is missing from .env")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
ANNOT_DIR = PROJECT_ROOT / "data" / "annotated"
RAW_DIR.mkdir(parents=True, exist_ok=True)
ANNOT_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", PROJECT_ROOT.name)
print("API key found: True (value hidden)")

## 2. Connect to YouTube API

A successful request tests the new key without displaying it. API quota limits and disabled comments may produce errors.

In [50]:
youtube = build("youtube", "v3", developerKey=API_KEY, cache_discovery=False)
try:
    test = youtube.videos().list(part="snippet", id="jNQXAC9IVRw").execute()
    print("YouTube API connected.", "Test videos returned:", len(test.get("items", [])))
except HttpError as exc:
    print("API error:", getattr(exc.resp, "status", "unknown"))
    raise

YouTube API connected. Test videos returned: 1


## 3. Enter 3–5 YouTube video URLs

Choose relevant public Vietnamese-language videos. Replace the examples below with real URLs you have selected. **No collection occurs until you add URLs.**

In [51]:
VIDEO_URLS = [
     "https://youtu.be/lQvsfDY6ef8",
    "https://youtu.be/fyrd-mpWxig",
]

MAX_COMMENTS_PER_VIDEO = 60  # Keep this small for a pilot.
PILOT_SAMPLE_SIZE = 40       # 30–50 is a useful second-pilot target.
RANDOM_SEED = 42

def video_id_from_url(value):
    s = value.strip()
    if re.fullmatch(r"[A-Za-z0-9_-]{11}", s):
        return s
    u = urlparse(s)
    host = (u.hostname or "").lower()
    if host in {"youtu.be", "www.youtu.be"}:
        candidate = u.path.strip("/").split("/")[0]
    elif host in {"youtube.com", "www.youtube.com", "m.youtube.com"}:
        if u.path == "/watch":
            candidate = parse_qs(u.query).get("v", [""])[0]
        elif u.path.startswith(("/shorts/", "/live/", "/embed/")):
            candidate = u.path.split("/")[2]
        else:
            candidate = ""
    else:
        candidate = ""
    if not re.fullmatch(r"[A-Za-z0-9_-]{11}", candidate):
        raise ValueError(f"Invalid YouTube video URL/ID: {value}")
    return candidate

VIDEO_IDS = list(dict.fromkeys(video_id_from_url(u) for u in VIDEO_URLS))
print("Videos selected:", len(VIDEO_IDS))

Videos selected: 2


## 4. Retrieve public top-level comments

`commentThreads.list` retrieves top-level comments only here; replies are **not** included. We store text, timestamps and source identifiers for private auditability, **not usernames or profile links**. Any video with comments disabled may fail and will be reported. Collection is bounded per video to limit requests.

In [52]:
def collect_video_comments(client, video_id, max_comments=60):
    collected = []
    page_token = None
    while len(collected) < max_comments:
        kwargs = dict(
            part="snippet", videoId=video_id,
            maxResults=min(100, max_comments - len(collected)),
            textFormat="plainText", order="time",
        )
        if page_token:
            kwargs["pageToken"] = page_token
        response = client.commentThreads().list(**kwargs).execute()
        for item in response.get("items", []):
            top = item.get("snippet", {}).get("topLevelComment", {})
            snip = top.get("snippet", {})
            collected.append({
                "comment_id": top.get("id", ""),
                "video_id": video_id,
                "source": "youtube",
                "source_url": f"https://www.youtube.com/watch?v={video_id}",
                "text": snip.get("textDisplay", ""),
                "published_at": snip.get("publishedAt", ""),
                "like_count": snip.get("likeCount", 0),
            })
        page_token = response.get("nextPageToken")
        if not page_token or not response.get("items"):
            break
    return collected

records = []
for vid in VIDEO_IDS:
    try:
        batch = collect_video_comments(youtube, vid, MAX_COMMENTS_PER_VIDEO)
        print(f"{vid}: {len(batch)} comments")
        records.extend(batch)
    except HttpError as exc:
        print(f"{vid}: API error {getattr(exc.resp, 'status', 'unknown')}; skipped")

RAW_COLUMNS = ["comment_id", "video_id", "source", "source_url", "text", "published_at", "like_count"]
raw_df = pd.DataFrame(records, columns=RAW_COLUMNS)
print("Total collected:", len(raw_df))

lQvsfDY6ef8: 60 comments
fyrd-mpWxig: 50 comments
Total collected: 110


## 5. Clean and deduplicate comments

Only minimal processing here: whitespace normalization, missing/blank removal, and duplicate removal. **Do not lowercase, strip emojis, remove punctuation, or alter spelling**: these are potential sarcasm cues and part of your later ablation.

A simple 4–100 whitespace-separated-token filter is **optional**. For Pilot 2, it is safer to inspect edge cases manually rather than discarding short sarcasm automatically.

In [53]:
def clean_comment(text):
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()

clean_df = raw_df.copy()
clean_df["text"] = clean_df["text"].map(clean_comment)
clean_df = clean_df[clean_df["text"].ne("")].copy()
clean_df = clean_df.drop_duplicates(subset=["comment_id"])
clean_df = clean_df.drop_duplicates(subset=["text"])
clean_df = clean_df.reset_index(drop=True)

print("Raw:", len(raw_df), "| Unique nonempty text:", len(clean_df))
if not clean_df.empty:
    display(clean_df[["video_id", "text"]].head(10))

Raw: 110 | Unique nonempty text: 109


,video_id,text
0,lQvsfDY6ef8,Khang rồng body đô kinh khủng nhưng face thì D...
1,lQvsfDY6ef8,NOte zenchung lại 1 đứa bú fame
2,lQvsfDY6ef8,"đm , trc có thg bên mma gạ tuấn hamez solo 11 ..."
3,lQvsfDY6ef8,Giọng khang rồng hơi hiền 😂
4,lQvsfDY6ef8,Khanh Rồng bá khí
5,lQvsfDY6ef8,Khang rồng : trần bình chỉ là bao kát biết đi ...
6,lQvsfDY6ef8,❤️
7,lQvsfDY6ef8,4:01 ai đó hãy làm meme đi
8,lQvsfDY6ef8,hết conten với hợi gym thì phải tìm mối mới rùi
9,lQvsfDY6ef8,Đừng bẻ ngang màn bạn ơi mình bị đau mắt :)))


## 6. Save private pilot collection

Files under `data/raw/` should remain ignored by Git. Check `.gitignore` before committing anything. Avoid showing or sharing raw files publicly if permission/ethical handling is unresolved.

In [54]:
raw_file = RAW_DIR / "pilot2_youtube_raw.csv"
clean_file = RAW_DIR / "pilot2_youtube_cleaned.csv"
raw_df.to_csv(raw_file, index=False, encoding="utf-8-sig")
clean_df.to_csv(clean_file, index=False, encoding="utf-8-sig")
print("Saved private raw and cleaned CSV files under data/raw/")

Saved private raw and cleaned CSV files under data/raw/


## 7. Prepare Pilot 2 annotation sheets

Both annotators receive the **same** randomly selected texts and IDs, in the **same order**, but in separate workbooks. They must **not** see each other's labels before agreement is calculated. `label` must be `0` (sincere), `1` (sarcasm/irony), or `UNCERTAIN` (annotation only); leave empty until annotated. Save independent source files; don't prefill label predictions. Refer to `annotation/annotation_guidelines.md`.

Some comments require context. Annotators can use `source_url` when needed and record missing-context notes. Be consistent about what context both annotators may access.

In [55]:
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation

# 1. File paths
project_root = Path.cwd().parent
input_path = project_root / "data/raw/pilot2_youtube_cleaned.csv"
output_path = project_root / "data/annotated/pilot2_annotation.xlsx"

output_path.parent.mkdir(parents=True, exist_ok=True)

# 2. Load and clean comments
df = pd.read_csv(input_path)

df = df.dropna(subset=["text"]).copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""]
df = df.drop_duplicates(subset=["text"])

# 3. Select 40 comments
df = df.sample(
    n=min(40, len(df)),
    random_state=42
).reset_index(drop=True)

# 4. Prepare annotation columns
annotation_df = pd.DataFrame({
    "ID": [f"{i:03d}" for i in range(1, len(df) + 1)],
    "text": df["text"],
    "label": "",
    "confidence": "",
    "notes": ""
})

# 5. Export two identical sheets
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for person in ["Person_1", "Person_2"]:
        annotation_df.to_excel(writer, sheet_name=person, index=False)

# 6. Add dropdowns and formatting
wb = load_workbook(output_path)

for ws in wb.worksheets:

    # Label dropdown
    label_dropdown = DataValidation(
        type="list",
        formula1='"Sincere,Sarcasm/Irony,Uncertain"',
        allow_blank=True
    )
    label_dropdown.error = "Please select a valid label."
    label_dropdown.errorTitle = "Invalid Label"
    label_dropdown.showErrorMessage = True
    ws.add_data_validation(label_dropdown)
    label_dropdown.add("C2:C1000")

    # Confidence dropdown
    confidence_dropdown = DataValidation(
        type="list",
        formula1='"1 - Low,2 - Medium,3 - High"',
        allow_blank=True
    )
    confidence_dropdown.error = "Please select a confidence level."
    confidence_dropdown.showErrorMessage = True
    ws.add_data_validation(confidence_dropdown)
    confidence_dropdown.add("D2:D1000")

    # Header formatting
    for cell in ws[1]:
        cell.fill = PatternFill(
            start_color="17365D",
            end_color="17365D",
            fill_type="solid"
        )
        cell.font = Font(color="FFFFFF", bold=True)
        cell.alignment = Alignment(horizontal="center")

    # Column widths
    ws.column_dimensions["A"].width = 14
    ws.column_dimensions["B"].width = 85
    ws.column_dimensions["C"].width = 22
    ws.column_dimensions["D"].width = 22
    ws.column_dimensions["E"].width = 55

    # Wrap long comments
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )
        ws.row_dimensions[row[0].row].height = 48

    # Freeze header and enable filtering
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

# 7. Save workbook
wb.save(output_path)

print(f"Workbook created: {output_path}")
print(f"Comments per annotator: {len(annotation_df)}")
print("Dropdown lists added to both sheets!")

Workbook created: c:\Users\84905\Documents\GitHub\FYP-Vietnamese-Sarcasm\data\annotated\pilot2_annotation.xlsx
Comments per annotator: 40
Dropdown lists added to both sheets!
